# Chapter 08 — Clustering Techniques: Live Examples

**Session 3 | Chapter 2 | Duration: ~10 min**

We demonstrate K-Means, Hierarchical Clustering, and DBSCAN on  
both synthetic and real customer data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Ready!')

## 1. K-Means: Elbow Method & Silhouette Score

In [ ]:
# Generate blob data with 4 true clusters
X_blobs, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=42)

# Run K-Means for k = 1 to 10
k_range = range(1, 11)
inertias, silhouette_scores = [], []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_blobs)
    inertias.append(km.inertia_)
    if k > 1:
        silhouette_scores.append(silhouette_score(X_blobs, labels))
    else:
        silhouette_scores.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(list(k_range), inertias, 'o-', color='#3498db', linewidth=2)
axes[0].axvline(4, color='#e74c3c', linestyle='--', label='True k=4')
axes[0].set_xlabel('Number of clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia (within-cluster sum of squares)', fontsize=12)
axes[0].set_title('Elbow Method — Look for the bend', fontsize=13)
axes[0].legend()

# Silhouette plot
axes[1].plot(list(k_range)[1:], silhouette_scores[1:], 's-', color='#2ecc71', linewidth=2)
axes[1].axvline(4, color='#e74c3c', linestyle='--', label='True k=4')
axes[1].set_xlabel('Number of clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score — Higher is better', fontsize=13)
axes[1].legend()

plt.suptitle('Choosing k: Elbow + Silhouette Agree on k=4', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize K-Means with k=4
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_blobs)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10', alpha=0.6, s=40)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='black', marker='X', s=200, zorder=10, label='Centroids')
ax.set_title(f'K-Means (k=4), Silhouette={silhouette_score(X_blobs, labels):.3f}', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Hierarchical Clustering with Dendrogram

In [ ]:
# Use a small subset for a readable dendrogram
X_small = X_blobs[:50]

Z = linkage(X_small, method='ward')

fig, ax = plt.subplots(figsize=(16, 5))
dendrogram(Z, ax=ax, leaf_rotation=90, leaf_font_size=8,
           color_threshold=5.5)
ax.axhline(y=5.5, color='red', linestyle='--', label='Cut here → 4 clusters')
ax.set_title('Hierarchical Clustering Dendrogram (Ward linkage)', fontsize=13)
ax.set_xlabel('Sample index')
ax.set_ylabel('Distance at which clusters merged')
ax.legend()
plt.tight_layout()
plt.show()

print('The red line shows where to cut the dendrogram to get 4 clusters.')
print('The height of the long vertical lines = natural gaps in the data.')

## 3. DBSCAN: Finding Arbitrary Shapes

K-Means fails on moon-shaped data. DBSCAN doesn't.

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.07, random_state=42)

# K-Means on moons (fails)
km_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km = km_moons.fit_predict(X_moons)

# DBSCAN on moons (works!)
scaler = StandardScaler()
X_moons_scaled = scaler.fit_transform(X_moons)
dbscan = DBSCAN(eps=0.25, min_samples=5)
labels_db = dbscan.fit_predict(X_moons_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# True
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='tab10', alpha=0.7, s=40)
axes[0].set_title('True groups (moons)', fontsize=12)

# K-Means
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_km, cmap='tab10', alpha=0.7, s=40)
axes[1].scatter(km_moons.cluster_centers_[:, 0], km_moons.cluster_centers_[:, 1],
                c='black', marker='X', s=200, zorder=10)
axes[1].set_title('K-Means (k=2) — FAILS on non-spherical data', fontsize=11)

# DBSCAN
colors_db = np.where(labels_db == -1, 'red', np.where(labels_db == 0, '#3498db', '#2ecc71'))
axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=colors_db, alpha=0.7, s=40)
n_noise = (labels_db == -1).sum()
axes[2].set_title(f'DBSCAN — SUCCEEDS! (noise={n_noise} points in red)', fontsize=11)

plt.suptitle('K-Means vs DBSCAN on Moon-Shaped Data', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('DBSCAN correctly identifies both moons without any shape assumptions!')

## 4. Real Data: Customer Segmentation

We create a synthetic customer dataset (spending patterns) and cluster them.

In [ ]:
np.random.seed(42)

# Simulate customer data: annual income and spending score (0-100)
n = 200
customers = pd.DataFrame({
    'annual_income_k': np.concatenate([
        np.random.normal(30, 5, 50),   # low income
        np.random.normal(60, 8, 70),   # medium income
        np.random.normal(90, 7, 80),   # high income
    ]),
    'spending_score': np.concatenate([
        np.random.normal(70, 8, 50),   # low income → high spending (credit?)
        np.random.normal(50, 10, 70),  # medium income → medium spending
        np.random.normal(30, 8, 80),   # high income → conservative spending
    ])
})

# Also add some high earners who spend a lot
customers = pd.concat([customers, pd.DataFrame({
    'annual_income_k': np.random.normal(85, 5, 30),
    'spending_score': np.random.normal(75, 7, 30),
})]).reset_index(drop=True)

print('Customer dataset:', customers.shape)
customers.describe().round(2)

In [ ]:
# Scale and cluster
X_cust = customers[['annual_income_k', 'spending_score']].values
X_cust_scaled = StandardScaler().fit_transform(X_cust)

km_cust = KMeans(n_clusters=4, n_init=10, random_state=42)
cust_labels = km_cust.fit_predict(X_cust_scaled)
customers['segment'] = cust_labels

# Profile the clusters
profile = customers.groupby('segment').agg(
    count=('segment', 'count'),
    avg_income=('annual_income_k', 'mean'),
    avg_spending=('spending_score', 'mean')
).round(1)
print('Cluster profiles:')
print(profile)

# Visualize
fig, ax = plt.subplots(figsize=(9, 7))
colors_cust = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
segment_names = ['Budget shoppers', 'Middle ground', 'Affluent savers', 'High earner–high spender']

for seg, (color, name) in enumerate(zip(colors_cust, segment_names)):
    mask = customers['segment'] == seg
    ax.scatter(customers.loc[mask, 'annual_income_k'],
               customers.loc[mask, 'spending_score'],
               color=color, label=f'Segment {seg}: {name}', s=60, alpha=0.7, edgecolors='white')

ax.set_xlabel('Annual Income (k€)', fontsize=12)
ax.set_ylabel('Spending Score', fontsize=12)
ax.set_title('Customer Segmentation with K-Means (k=4)', fontsize=14)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('\nEach segment has a distinct income-spending profile.')
print('A marketing team could tailor campaigns for each group!')